# 00 Load Library
---


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib


# 01 Inisialisasi Path dan Konsolidasi Fitur Wilayah
---
## Konfigurasi Data Makro
---
### Pembacaan Enam Dimensi Indikator Wilayah
---
Memuat indikator ekonomi makro, demografi, tarif sewa, dan kepadatan usaha pada 5 kabupaten/kota DIY.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'analisis' else Path.cwd()
CLEANED_DIR = BASE_DIR / 'data' / 'cleaned'
HASIL_DIR = BASE_DIR / 'outputs' / 'hasil'
MODELS_SAVED_DIR = BASE_DIR / 'models' / 'saved'
MODELS_SAVED_DIR.mkdir(parents=True, exist_ok=True)
df_ekonomi = pd.read_csv(CLEANED_DIR / 'dataset2_bps_makro_cleaned.csv')
df_sewa = pd.read_csv(CLEANED_DIR / 'biaya_operasional.csv')
df_sibakul = pd.read_csv(CLEANED_DIR / 'sibakul_umkm_bersih.csv')
df_kompetitor = pd.read_csv(CLEANED_DIR / 'kompetitor_per_wilayah.csv')


In [ ]:
komp_wil = df_kompetitor.groupby('wilayah').size().reset_index(name='total_poi')
df_clustering = df_ekonomi.merge(df_sewa, on='wilayah').merge(df_sibakul, on='wilayah').merge(komp_wil, on='wilayah')
df_clustering['rasio_kompetitor'] = (df_clustering['total_poi'] / (df_clustering['kepadatan_jiwa_km2'] * 0.1)).round(2)
feature_cols = ['pdrb_per_kapita_ribu_rp', 'pengeluaran_per_kapita_rp', 'kepadatan_jiwa_km2', 'tarif_sewa_per_m2_tahun', 'rasio_kompetitor', 'densitas_per_1000']
X = df_clustering[feature_cols].copy()
X


# 02 Standardisasi Fitur Z-Score
---
## Normalisasi Distribusi
---
### Penyeragaman Skala Fitur Menjadi Zero Mean dan Unit Variance
---
Standardisasi Z-Score wajib dilakukan agar variabel berskala nominal jutaan tidak mendominasi jarak Euclidean.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, MODELS_SAVED_DIR / 'scaler_clustering.joblib')
pd.DataFrame(X_scaled, columns=feature_cols, index=df_clustering['wilayah'])


# 03 Pemodelan K-Means Clustering K=3
---
## Optimasi Algoritma Lloyd
---
### Segmentasi Tiga Profil Tipologi Wilayah
---
Meminimalkan inersia WCSS untuk memisahkan wilayah ke dalam tiga klaster homogen.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df_clustering['cluster'] = clusters
joblib.dump(kmeans, MODELS_SAVED_DIR / 'kmeans_wilayah.joblib')
df_clustering[['wilayah', 'cluster']]


# 04 Reduksi Dimensi Principal Component Analysis PCA 2D
---
## Dekomposisi Nilai Eigen
---
### Proyeksi Enam Dimensi ke Bidang Kartesius Dua Dimensi
---
Mengukur proporsi variansi data asli yang berhasil dipertahankan pada bidang proyeksi dua dimensi.


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_clustering['pca_1'] = X_pca[:, 0].round(3)
df_clustering['pca_2'] = X_pca[:, 1].round(3)
joblib.dump(pca, MODELS_SAVED_DIR / 'pca_wilayah.joblib')
var_ratio = pca.explained_variance_ratio_
total_var = var_ratio.sum() * 100
info_loss = 100 - total_var
print(f'PC1: {var_ratio[0]*100:.2f}% | PC2: {var_ratio[1]*100:.2f}% | Total: {total_var:.2f}% | Loss: {info_loss:.2f}%')


# 05 Karakterisasi Profil Klaster dan Ekspor
---
## Pelabelan Tipologi Ekonomi
---
### Penyimpanan Hasil Clustering Wilayah ke CSV
---
Menetapkan label karakteristik pasar pada masing-masing klaster dan mengekspor hasilnya.


In [ ]:
label_map = {
    0: 'Urban Padat & Biaya Tinggi',
    1: 'Semi-Urban Bertumbuh (Sweet Spot)',
    2: 'Rural Potensial & Biaya Rendah'
}
df_clustering['karakteristik_pasar'] = df_clustering['cluster'].map(label_map)
cols_export_cluster = ['wilayah', 'cluster', 'karakteristik_pasar', 'pca_1', 'pca_2', 'tarif_sewa_per_m2_tahun', 'pdrb_per_kapita_ribu_rp']
output_cluster = HASIL_DIR / 'hasil_clustering_wilayah.csv'
df_clustering[cols_export_cluster].to_csv(output_cluster, index=False)
print('Hasil clustering Pilar 2 berhasil disimpan di:', output_cluster)
df_clustering[cols_export_cluster]
